# RandLA-Net Using Open3D-ML

https://github.com/isl-org/Open3D-ML?tab=readme-ov-file#semantic-segmentation

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import open3d.ml as _ml3d
import open3d.ml.torch as ml3d
import ml3d.torch as ml3d


cfg_file = "/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/configs/randlanet_semantickitti.yml"
cfg = _ml3d.utils.Config.load_from_file(cfg_file)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
model = ml3d.models.RandLANet(**cfg.model)
cfg.dataset['dataset_path'] = '/media/arthur/HDD/Datasets/Point Clouds/SemanticKitti/raw/'
dataset = _ml3d.datasets.SemanticKITTI(cfg.dataset.pop('dataset_path', None), **cfg.dataset)
pipeline = ml3d.pipelines.SemanticSegmentation(model, dataset=dataset, device="gpu", **cfg.pipeline)

In [4]:
# download the weights.
ckpt_folder = "./logs/"
os.makedirs(ckpt_folder, exist_ok=True)
ckpt_path = ckpt_folder + "randlanet_semantickitti_202201071330utc.pth"
randlanet_url = "https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth"
if not os.path.exists(ckpt_path):
    cmd = "wget {} -O {}".format(randlanet_url, ckpt_path)
    os.system(cmd)

In [5]:
# load the parameters.
pipeline.load_ckpt(ckpt_path=ckpt_path)

In [6]:
test_split = dataset.get_split("test")
data = test_split.get_data(0)

In [7]:
data

{'point': array([[ 1.35929756e+01,  7.97351729e-03,  6.69000268e-01],
        [ 1.35728445e+01,  5.08311652e-02,  6.68001473e-01],
        [ 1.35917816e+01,  7.17616528e-02,  6.69002056e-01],
        ...,
        [ 7.93910265e+00, -2.63877869e+00, -3.77945089e+00],
        [ 7.96492624e+00, -2.61984015e+00, -3.78844428e+00],
        [ 7.98275280e+00, -2.59890866e+00, -3.79343700e+00]], dtype=float32),
 'feat': None,
 'label': array([0, 0, 0, ..., 0, 0, 0], dtype=int32)}

In [8]:
# run inference on a single example.
# returns dict with 'predict_labels' and 'predict_scores'.
result = pipeline.run_inference(data)

test 0/1: 100%|█████████▉| 79554/79845 [00:02<00:00, 28949.97it/s]/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append(np.nanmean(accs))
/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/torch/modules/metrics/semseg_metric.py:87: RuntimeWarning: Mean of empty slice
  ious.append(np.nanmean(ious))


In [9]:
import torch
from torch.utils.data import DataLoader

from ml3d.torch.dataloaders import get_sampler, TorchDataloader
from ml3d.datasets import InferenceDummySplit


batcher = pipeline.get_batcher("cpu")
infer_dataset = InferenceDummySplit(data)
pipeline.dataset_split = infer_dataset

infer_sampler = infer_dataset.sampler
infer_split = TorchDataloader(dataset=infer_dataset,
                                preprocess=model.preprocess,
                                transform=model.transform,
                                sampler=infer_sampler,
                                use_cache=False)

infer_loader = DataLoader(infer_split,
                            batch_size=2,
                            sampler=get_sampler(infer_sampler),
                            collate_fn=batcher.collate_fn)

model.trans_point_sampler = infer_sampler.get_point_sampler()

In [10]:
data['point'].shape

(127405, 3)

In [11]:
curr_cloud_id = -1
test_probs = []
ori_test_probs = []
ori_test_labels = []

with torch.no_grad():
    for unused_step, inputs in enumerate(infer_loader):
        results = model(inputs['data'])
        break

In [12]:
len(inputs['data']['coords'])

4

In [13]:
inputs['data']['coords'][3].shape

torch.Size([2, 704, 3])

In [14]:
inputs['data']['neighbor_indices']

[tensor([[[    0,  8360, 14468,  ..., 40097, 32811, 25370],
          [    1, 19900,  6158,  ..., 31171, 16408, 37703],
          [    2, 40834,  6572,  ..., 26968, 24401, 23650],
          ...,
          [45053, 44554, 35271,  ...,  8094, 23376, 12294],
          [45054,  8426, 21209,  ...,   325, 19380, 25352],
          [45055, 37629,  8106,  ..., 33703, 40884, 12902]],
 
         [[    0, 29024, 37142,  ...,  9493, 43061,   783],
          [    1, 34035, 28047,  ..., 23011,  3024, 32230],
          [    2,  7283, 43337,  ...,  3471,  9167, 39229],
          ...,
          [45053,  1220, 38744,  ..., 32903, 13159, 34055],
          [45054, 18425, 29206,  ..., 41554, 38649, 34522],
          [45055, 22292, 15369,  ..., 21586, 44763, 15708]]]),
 tensor([[[    0,  8360, 10213,  ...,  4790,  3051,  2649],
          [    1,  6158,  7770,  ...,  5856,  1599, 10437],
          [    2,  6572,  9830,  ...,  3385,  2750,  6796],
          ...,
          [11261,  2104,  8112,  ..., 10798,  233

In [15]:
inputs['data']['sub_idx']

[tensor([[[    0,  8360, 14468,  ..., 40097, 32811, 25370],
          [    1, 19900,  6158,  ..., 31171, 16408, 37703],
          [    2, 40834,  6572,  ..., 26968, 24401, 23650],
          ...,
          [11261, 35516, 44781,  ..., 32470, 29009, 29989],
          [11262, 27560,  1086,  ..., 24877, 35351, 34634],
          [11263, 22396, 29095,  ..., 16553,  7501, 42802]],
 
         [[    0, 29024, 37142,  ...,  9493, 43061,   783],
          [    1, 34035, 28047,  ..., 23011,  3024, 32230],
          [    2,  7283, 43337,  ...,  3471,  9167, 39229],
          ...,
          [11261, 25077, 32348,  ..., 34509,  5629, 25041],
          [11262,  4711, 20191,  ...,  5894, 20295, 43812],
          [11263, 28886,   150,  ..., 18872, 23115, 35709]]]),
 tensor([[[    0,  8360, 10213,  ...,  4790,  3051,  2649],
          [    1,  6158,  7770,  ...,  5856,  1599, 10437],
          [    2,  6572,  9830,  ...,  3385,  2750,  6796],
          ...,
          [ 2813,  6467, 10401,  ...,  6313,  515

In [17]:
inputs["data"]["features"].shape

torch.Size([2, 45056, 3])